# 00 — Environment check: Apple Silicon (MPS)

Verifies the device, loads ModernBERT-large, and runs forward + backward + optimizer steps with LoRA and with full fine-tuning at training shape. Reports peak memory.

All logic lives in `jevlite.envcheck`; this notebook only differs from its twin in the first code cell.

## 0. Install (Kaggle / Colab)

On Kaggle: Settings → Internet → On, Accelerator → GPU T4 x2.

In [ ]:
# Kaggle / Colab: install the code from GitHub at a fixed commit. A no-op on a local checkout.
import os, sys

COMMIT = "main"   # set a commit hash for a reproducible run
ON_CLOUD = "KAGGLE_KERNEL_RUN_TYPE" in os.environ or "COLAB_RELEASE_TAG" in os.environ
if ON_CLOUD:
    !rm -rf /tmp/JEV_Judge && git clone -q https://github.com/Nefelim584/JEV_Judge.git /tmp/JEV_Judge
    !cd /tmp/JEV_Judge && git checkout -q {COMMIT} && git log -1 --format='%h %ad %s' --date=short
    !pip install -q --no-deps --ignore-requires-python -e /tmp/JEV_Judge
    !pip install -q loguru peft bitsandbytes
    !pip uninstall -y -q torchao  # the image has torchao 0.10; recent peft refuses < 0.16 instead of skipping it
    sys.path.insert(0, "/tmp/JEV_Judge/src")  # the running kernel does not pick up a fresh editable install
!python --version

## 1. Environment and config

The only stack-specific cell. Env vars are applied before torch is imported.

In [1]:
STACK = "apple_silicon"

from jevlite.config import apply_env, load_config, seed_everything
from jevlite.log import logger, setup_logging

setup_logging()
cfg = load_config(STACK)
apply_env(cfg)
logger.info("stack {}: config loaded", STACK)

## 2. Device report

In [2]:
import pprint

import torch
import transformers

from jevlite.device import device_report, resolve_runtime

rt = resolve_runtime(cfg)
report = device_report(rt) | {"transformers": transformers.__version__}
logger.info("device report:\n{}", pprint.pformat(report))
_ = seed_everything(cfg["seed"])

/Users/nefelim584/PycharmProjects/JEV_Judge/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'python': '3.14.0',
 'torch': '2.14.1',
 'device': 'mps',
 'precision': 'fp32',
 'attn_implementation': 'sdpa',
 'optimizer': 'adamw',
 'mps_available': True,
 'cuda_available': False,
 'bitsandbytes_installed': False,
 'flash_attn_installed': False,
 'mps_recommended_max_gb': 11.8,
 'transformers': '5.18.0'}


## 3. Request smoke test

Sample request → schema → encoding → collate → encoder → grouped logits, on the device. The probe head is random, so only shapes and masking are checked.

In [3]:
from jevlite.envcheck import request_forward_check

smoke = request_forward_check(cfg, rt)
logger.info("request smoke test:\n{}", pprint.pformat(smoke))
assert smoke["ok"]

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 170/170 [00:00<00:00, 5857.63it/s]


[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'rows': 10,
 'seq_len': 59,
 'grouped_shape': (3, 5),
 'padded_candidate_mass': 0.0,
 'row_sums': [1.0000001192092896, 1.0, 1.0],
 'ok': True}


## 4. Training step: LoRA and full fine-tuning

Uses `train.per_device_batch_size` and `encoding.max_len` from the config, the configured precision (AMP + GradScaler where applicable) and optimizer.

In [4]:
from jevlite.envcheck import train_step_check

batch_size = cfg["train"]["per_device_batch_size"]
seq_len = cfg["encoding"]["max_len"]
results = []
for mode in ("lora", "full"):
    r = train_step_check(cfg, rt, mode=mode, batch_size=batch_size, seq_len=seq_len)
    logger.info("{} training step:\n{}", mode, pprint.pformat(r))
    assert r["params_updated"], f"{mode}: parameters did not change"
    results.append(r)

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 170/170 [00:00<00:00, 12852.51it/s]


[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'mode': 'lora',
 'precision': 'fp32',
 'optimizer': 'adamw',
 'batch_size': 8,
 'seq_len': 512,
 'trainable_params': 7196672,
 'total_params': 401978368,
 'trainable_pct': 1.79,
 'losses': [0.8801, 0.7215, 0.6086],
 'params_updated': True,
 'first_step_s': 5.05,
 'step_s': 5.41,
 'peak_memory_gb': 2.04}


Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 170/170 [00:00<00:00, 14361.45it/s]


[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-large
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


{'mode': 'full',
 'precision': 'fp32',
 'optimizer': 'adamw',
 'batch_size': 8,
 'seq_len': 512,
 'trainable_params': 394781696,
 'total_params': 394781696,
 'trainable_pct': 100.0,
 'losses': [0.7738, 0.6053, 0.599],
 'params_updated': True,
 'first_step_s': 5.99,
 'step_s': 6.04,
 'peak_memory_gb': 5.92}


## 5. Summary

In [5]:
cols = ["mode", "precision", "optimizer", "batch_size", "seq_len", "trainable_pct", "step_s", "peak_memory_gb"]
lines = [" | ".join(cols)] + [" | ".join(str(r[c]) for c in cols) for r in results]
logger.info("summary:\n{}", "\n".join(lines))
logger.info("device: {}, default tuning mode for this stack: {}", report["device"], cfg["tuning"]["mode"])

mode | precision | optimizer | batch_size | seq_len | trainable_pct | step_s | peak_memory_gb
lora | fp32 | adamw | 8 | 512 | 1.79 | 5.41 | 2.04
full | fp32 | adamw | 8 | 512 | 100.0 | 6.04 | 5.92

device: mps, default tuning mode for this stack: lora
